In [3]:
import pandas as pd

# adjust paths to your actual locations
meta = pd.read_csv(
    "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/cxr/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv.gz"
)
cohort = pd.read_parquet(
    "../data/cohort/sepsis_labels.parquet"
)
cohort = cohort[cohort["excluded_reason"].isna()]

subset = meta[meta["subject_id"].isin(cohort["subject_id"])]
print(f"{len(subset)} CXR images needed for {cohort['subject_id'].nunique()} cohort subjects "
      f"(out of {len(meta)} total in the full dataset)")

with open("cxr_urls.txt", "w") as f:
    for _, r in subset.iterrows():
        s = str(int(r.subject_id))
        f.write(
            f"https://physionet.org/files/mimic-cxr-jpg/2.0.0/files/"
            f"p{s[:2]}/p{s}/s{int(r.study_id)}/{r.dicom_id}.jpg\n"
        )

150354 CXR images needed for 52560 cohort subjects (out of 377110 total in the full dataset)


In [ ]:
## Script to download


# cd "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction"

# wget -c -np \
#   --user fluuvys --ask-password \
#   -i cxr_urls.txt \
#   -x -nH --cut-dirs=3 \
#   -P "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/physionet.org/files/mimic-cxr-jpg/2.0.0"


cat cxr_urls.txt | parallel -j 6 --bar \
  wget -c -np --user fluuvys --password @Fluuvyshust2005 \
    -x -nH --cut-dirs=3 \
    -P /home/fluuvys-main/sepsis_proj/Data/physionet.org/files/mimic-cxr-jpg/2.0.0 \
    {}
    

In [1]:
import pandas as pd

meta = pd.read_csv("/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/cxr/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv.gz")
cohort = pd.read_parquet("../data/cohort/sepsis_labels.parquet")
cohort = cohort[cohort["excluded_reason"].isna()]
subset = meta[meta["subject_id"].isin(cohort["subject_id"])]

with open("cxr_urls_aria2.txt", "w") as f:
    for _, r in subset.iterrows():
        s = str(int(r.subject_id))
        url = (f"https://physionet.org/files/mimic-cxr-jpg/2.0.0/files/"
               f"p{s[:2]}/p{s}/s{int(r.study_id)}/{r.dicom_id}.jpg")
        out = f"files/p{s[:2]}/p{s}/s{int(r.study_id)}/{r.dicom_id}.jpg"
        f.write(f"{url}\n  out={out}\n")

In [ ]:
cat > generate_gcs_sync.py << 'EOF'
import pandas as pd

meta = pd.read_csv("/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/cxr/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv.gz")  # adjust path if needed
cohort = pd.read_parquet("../data/cohort/sepsis_labels.parquet")
cohort = cohort[cohort["excluded_reason"].isna()]

# only subjects that actually appear in the CXR metadata -- this is the
# real fix, avoids ~44,700 expected-but-noisy failures for subjects with
# zero CXR studies
subjects_with_cxr = sorted(
    int(s) for s in set(meta["subject_id"]) & set(cohort["subject_id"])
)

base = "/home/fluuvys-main/sepsis_proj/Data/physionet.org/files/mimic-cxr-jpg/2.0.0/files"
project_id = "project-3a8c4ca8-da13-47aa-945"

with open("gcs_sync_commands.sh", "w") as f:
    for s in subjects_with_cxr:
        shard = str(s)[:2]
        f.write(
            f"gsutil -u {project_id} -m rsync -r "
            f"'gs://mimic-cxr-jpg-2.1.0.physionet.org/files/p{shard}/p{s}' "
            f"'{base}/p{shard}/p{s}'\n"
        )
print(f"{len(subjects_with_cxr)} subjects with CXR data -> "
      f"{len(subjects_with_cxr)} sync commands written")
EOF

python3 generate_gcs_sync.py

In [ ]:


# python cxr_linking.py \
#   --mimic_cxr_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/cxr/files/mimic-cxr-jpg/2.0.0" \
#   --mimic_iv_hosp_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/mimic-iv-3.1/hosp" \
#   --sepsis_labels_path "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort/sepsis_labels.parquet" \
#   --out_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort" \
#   --resized_image_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort/cxr/resized"

In [ ]:
cd preprocessing
rm data/cache/*.parquet

# 1. Full rerun, both stages
python label_sepsis3.py \
  --mimic_hosp_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/mimic-iv-3.1/hosp" \
  --mimic_icu_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/mimic-iv-3.1/icu" \
  --out_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort" \
  --splits_path "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/splits/subject_splits.parquet" \
  --cache_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cache"
# 2. ehr_extraction.py -- new sepsis_labels.parquet + its own reader fix
python ehr_extraction.py \
  --data_root "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/mimic-iv-3.1" \
  --labels_path "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort/sepsis_labels.parquet" \
  --output "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort/ehr_timeseries.parquet" \
  --cache_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cache"

# 3. cxr_linking.py -- new sepsis_labels.parquet + its own reader fix
python cxr_linking.py \
  --mimic_cxr_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/cxr/files/mimic-cxr-jpg/2.0.0" \
  --mimic_iv_hosp_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/mimic-iv-3.1/hosp" \
  --sepsis_labels_path "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort/sepsis_labels.parquet" \
  --out_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort" \
  --resized_image_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort/cxr/resized"

# 4. notes_extraction.py -- unchanged code, but rerun for the corrected cohort/icu_intime
python notes_extraction.py \
  --mimic_note_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Data/physionet.org/files/mimic-iv-note/2.2/note" \
  --sepsis_labels_path "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort/sepsis_labels.parquet" \
  --out_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cohort" \
  --cache_dir "/home/fluuvys-main/Research/Multi modal sepsis prediction/Multi-Modal-Sepsis-Prediction/data/cache"